# E1.5 · Evaluation output as audit evidence

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

---

**Risk.** Accepting a vendor's best-of-k demo as assurance; mistaking schema conformance for accuracy.

**Control.** Read an eval report properly: execution-verified results, reliability across all attempts, trajectory scoring, judge independence.

**This lab.** Turn an eval report into audit evidence — and find how it could mislead you.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness, OSCAL |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E1.5"))

Evaluation output as audit evidence — the bridge between B2.10 and the evidence pack. It only works if you present the right number.

In [ ]:
from cybercommons import evalkit, grc
import time

truths = {f"q{i}": evalkit.Truth(f"q{i}",
              ["CWE-89", "CWE-78", "CWE-22", "CWE-798"][i % 4],
              f"{['CWE-89','CWE-78','CWE-22','CWE-798'][i % 4]}/{i}.py")
          for i in range(1, 13)}
answers = {q: '{"qid":"%s","cwe":"%s","file":"%s","rationale":"untrusted input reaches the sink"}'
                % (q, t.cwe if int(q[1:]) % 3 else "CWE-89", t.file)
           for q, t in truths.items()}
rep = evalkit.evaluate(answers, truths)
print(rep.render())

Now the part that decides whether this is evidence or a marketing slide.

In [ ]:
now = time.time()
test = grc.ControlTest("EV-2", passed=rep.expert_accuracy >= 0.80,
                       evidence=f"expert accuracy {rep.expert_accuracy:.4f} over "
                                f"{rep.total} held-out questions",
                       tested_at=now, valid_for_days=30)
print(f"EV-2 → {test.state(now)}  ({test.evidence})")
print("\nWhat makes this auditable:")
for line in ["the key was held out — the harness never saw it",
             "the number reported is accuracy, not conformance",
             "the sample size is stated",
             "it expires in 30 days, so it cannot silently age into a claim"]:
    print("  ·", line)

### Expect

The report prints conformance 1.0 alongside a materially lower expert accuracy, and the control test records the accuracy figure with a 30-day validity window.

### Your turn

Take an eval number your organisation has quoted externally. Was it conformance, pass-rate on a public set, or accuracy against a held-out key? Only the third is evidence.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E1.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*